# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** Can content items in the warehouse be grouped into a few performance archetypes, using only CTR, average position and impressions, that still hold up on clients the model never saw — and can each archetype be mapped to a sensible review action?

**Decision it supports:** Where should an editorial / SEO team spend limited review hours next quarter: fix titles, look closer at a page, or leave it alone?

**Who acts:** Content operations leads (assign the work) and SEO strategists (do the title / meta / linking work).

#### Cost of a wrong call.

- Prune a page that was really earning traffic → traffic lost overnight.
- Rewrite a whole article when a title tweak was enough → expensive hours wasted.

In [1]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
warnings.filterwarnings('ignore')

SEED = 42
SIL_N = 20000
np.random.seed(SEED)
os.makedirs('work/outputs/figures', exist_ok=True)

def find(name):
    for d in ['work/outputs', '/content/drive/MyDrive', '.']:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found - run w07 first or upload it next to this notebook.")

ACTION_MAP = {
    'High-Reach Opportunity':  ('FIX_TITLE_META_PRIORITIZED',  'RC_HIGH_REACH_CONFIRMED_GAP'),
    'Low-Visibility / Buried': ('REVIEW_FOR_PRUNE_OR_REWORK',  'RC_WORST_POSITION_LOW_SIGNAL'),
    'Steady / Low Signal':     ('LOW_PRIORITY_REVIEW',         'RC_MODEST_ALL_AXES'),
    'Already Converting':      ('MONITOR_ONLY',                'RC_HIGH_CTR_NO_GAP'),
}
for a, (act, rc) in ACTION_MAP.items():
    print(f"{a:26s} -> {act:28s} ({rc})")

High-Reach Opportunity     -> FIX_TITLE_META_PRIORITIZED   (RC_HIGH_REACH_CONFIRMED_GAP)
Low-Visibility / Buried    -> REVIEW_FOR_PRUNE_OR_REWORK   (RC_WORST_POSITION_LOW_SIGNAL)
Steady / Low Signal        -> LOW_PRIORITY_REVIEW          (RC_MODEST_ALL_AXES)
Already Converting         -> MONITOR_ONLY                 (RC_HIGH_CTR_NO_GAP)


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank internship warehouse, build v20260703 (pseudonymized, gated).

**Table used:** fact_content_daily_performance only — Search Console columns: impressions, clicks, average position.

**Window:** Report dates 2025-01-27 to 2026-06-30. Each item is summed over its own available history (history depth differs by client) — not a Jan–Jun 2026 window.

One row = one content item (content_hash_id), one aggregate over its history.

**Kept:** Daily rows with impressions > 0, then items with at least 100 impressions in total → 182,135 items from 59 clients (the release lists 104 clients; the rest have too little search history to pass the filter).

**Excluded:**
- GA4 columns — zero-filled before each client's GA4 start; not needed for CTR / position / volume.
- fact_content_query_90d — its 90-day window overlaps the most recent months; skipping it avoids window-overlap leakage.
- dim_content / dim_clients — no field from them is a feature.
- Items under 100 impressions — CTR ratios are too noisy to act on (w04).

**Unit note:** In the warehouse aggregate, CTR is a fraction (0.0023 = 0.23%) — do not mix it with the ×100-percentage starter CSV.

**IDs (content_hash_id, client_hash_id) are pseudonyms:** grouping and splitting only, never features.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv(find('w07_action_playbook_queue.csv'))
summary = json.load(open(find('w07_playbook_summary.json')))

print(f"Rows: {len(df):,}   Clients: {df['client_hash_id'].nunique()}")
print(f"Impressions per item: min {df['impressions'].min():.0f}, median {df['impressions'].median():.0f}, max {df['impressions'].max():,.0f}")

dups = df['content_hash_id'].duplicated().sum()
print(f"Duplicate content_hash_id rows: {dups}")
print(f"Missing values in ctr / avg_position / impressions: "
      f"{int(df[['ctr','avg_position','impressions']].isna().sum().sum())}")
print(f"Items with avg_position == 0: {(df['avg_position'] == 0).sum()}")

assert len(df) == summary['n_items'] == 182135
assert df['client_hash_id'].nunique() == 59
assert dups == 0 and df['impressions'].min() >= 100

Mounted at /content/drive
Rows: 182,135   Clients: 59
Impressions per item: min 100, median 1513, max 2,902,616
Duplicate content_hash_id rows: 0
Missing values in ctr / avg_position / impressions: 0
Items with avg_position == 0: 1


In [4]:
try:
    from google.colab import userdata
    import duckdb
    con = duckdb.connect()
    con.execute("CREATE SECRET (TYPE huggingface, TOKEN ?)", [userdata.get('HF_TOKEN')])
    base = 'hf://datasets/FlyRank/internship-warehouse'
    print(con.sql(f"""SELECT COUNT(*) AS rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day
                      FROM read_parquet('{base}/fact_content_daily_performance/**/*.parquet')""").df())
    print(con.sql(f"SELECT COUNT(*) AS clients FROM read_parquet('{base}/dim_clients.parquet')").df())
except Exception as e:
    print("Warehouse check skipped (no Colab secret / no access):", type(e).__name__)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       rows  first_day   last_day
0  78835655 2025-01-27 2026-06-30
   clients
0      104


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions:**
- CTR, position and volume together describe how a page behaves in search — not what it says.
- Clients differ, so a model must be tested on clients it did not train on.

**Features (3):** ctr, avg_position, impressions_log = log10(impressions + 1). No missing values, so no fills. Standardized on training clients only.

**Label:** None — unsupervised. The baseline score is a comparison ranking, never an input.

**Baseline (w04):**
score = (expected_ctr − actual_ctr) × impressions,
clipped at zero. expected_ctr is the average CTR of items in the same position quartile.

**Model:** K-Means, random_state=42, n_init=10. k compared from 3 to 9.

**Validation design:** 80/20 split grouped by client: 47 train, 12 test.

**Leakage checks:**
- (1) no baseline-derived column is a feature,
- (2) shared-input check,
- (3) window check.

In [5]:
df = df.sort_values('score', ascending=False, kind='stable').reset_index(drop=True)
df['impressions_log'] = np.log10(df['impressions'] + 1)
FEATURES = ['ctr', 'avg_position', 'impressions_log']
X = df[FEATURES].values

clients = df['client_hash_id'].unique()
shuffled = np.random.RandomState(SEED).permutation(clients)
train_clients = set(shuffled[:int(0.8 * len(shuffled))])
train_mask = df['client_hash_id'].isin(train_clients).values
test_mask = ~train_mask

print(f"Train: {len(train_clients)} clients, {train_mask.sum():,} items")
print(f"Test:  {len(clients) - len(train_clients)} clients, {test_mask.sum():,} items")

assert not (set(df.loc[train_mask, 'client_hash_id']) & set(df.loc[test_mask, 'client_hash_id']))
assert train_mask.sum() == 161153 and test_mask.sum() == 20982

scaler = StandardScaler().fit(X[train_mask])
Xs_train, Xs_test = scaler.transform(X[train_mask]), scaler.transform(X[test_mask])

Train: 47 clients, 161,153 items
Test:  12 clients, 20,982 items


In [6]:
LOCKED_K = {3: (0.356, 0.269), 4: (0.378, 0.336), 5: (0.379, 0.336), 6: (0.315, 0.267),
            7: (0.316, 0.267), 8: (0.310, 0.264), 9: (0.313, 0.275)}
rows = []
for k in range(3, 10):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10, max_iter=300).fit(Xs_train)
    tr = silhouette_score(Xs_train, km.labels_, sample_size=SIL_N, random_state=SEED)
    te = silhouette_score(Xs_test, km.predict(Xs_test), sample_size=SIL_N, random_state=SEED)
    rows.append((k, tr, te, *LOCKED_K[k]))
ksel = pd.DataFrame(rows, columns=['k', 'train_sil', 'test_sil', 'locked_train', 'locked_test'])
print(ksel.round(3).to_string(index=False))

best = ksel['test_sil'].max()
K = int(ksel.loc[ksel['test_sil'] >= best - 0.005, 'k'].min())
print(f"\nBest held-out silhouette: {best:.3f}. Simplest k within 0.005: k = {K}")
assert K == 4
assert (ksel['test_sil'] - ksel['locked_test']).abs().max() < 0.02

 k  train_sil  test_sil  locked_train  locked_test
 3      0.355     0.268         0.356        0.269
 4      0.377     0.336         0.378        0.336
 5      0.377     0.335         0.379        0.336
 6      0.315     0.267         0.315        0.267
 7      0.315     0.267         0.316        0.267
 8      0.309     0.263         0.310        0.264
 9      0.312     0.275         0.313        0.275

Best held-out silhouette: 0.336. Simplest k within 0.005: k = 4


In [7]:
scaler_all = StandardScaler().fit(X)
km_all = KMeans(n_clusters=K, random_state=SEED, n_init=10, max_iter=300).fit(scaler_all.transform(X))
df['cluster'] = km_all.labels_

prof = df.groupby('cluster').agg(n=('score', 'size'), mean_ctr=('ctr', 'mean'),
                                 mean_position=('avg_position', 'mean'),
                                 mean_impressions=('impressions', 'mean'),
                                 mean_score=('score', 'mean')).reset_index()
prof['ctr_rank'] = prof['mean_ctr'].rank(ascending=False)
prof['position_rank'] = prof['mean_position'].rank(ascending=True)
prof['impressions_rank'] = prof['mean_impressions'].rank(ascending=False)
prof['score_rank'] = prof['mean_score'].rank(ascending=False)

def name_archetype(r):
    if r['ctr_rank'] == 1 and r['score_rank'] == prof['score_rank'].max(): return 'Already Converting'
    if r['impressions_rank'] == 1 and r['score_rank'] == 1:                return 'High-Reach Opportunity'
    if r['position_rank'] == prof['position_rank'].max():                  return 'Low-Visibility / Buried'
    return 'Steady / Low Signal'

prof['archetype'] = prof.apply(name_archetype, axis=1)
assert prof['archetype'].nunique() == len(prof)
df['archetype_refit'] = df['cluster'].map(dict(zip(prof['cluster'], prof['archetype'])))

agree = (df['archetype_refit'] == df['archetype']).mean()
print(prof[['cluster', 'archetype', 'n', 'mean_ctr', 'mean_position', 'mean_impressions', 'mean_score']].round(4).to_string(index=False))
print(f"\nRefit vs saved w07 archetypes: {100 * agree:.2f}% agree")
assert agree > 0.99

 cluster               archetype     n  mean_ctr  mean_position  mean_impressions  mean_score
       0     Steady / Low Signal 77858    0.0024        13.7327          807.0961      1.2802
       1 Low-Visibility / Buried 29215    0.0009        45.7844         1458.3467      1.6070
       2  High-Reach Opportunity 67736    0.0032        11.7661        24073.9146     32.9690
       3      Already Converting  7326    0.0209        10.2509         3355.3490      0.0000

Refit vs saved w07 archetypes: 99.98% agree


In [8]:
# Leakage hunt: no banned column as a feature, shared-input correlation, window sanity check

BANNED = ['score', 'ctr_gap', 'expected_ctr', 'action_label', 'reason_code', 'position_bucket',
          'playbook_action', 'playbook_reason_code', 'priority_rank', 'archetype', 'archetype_refit', 'cluster']
assert not set(FEATURES) & set(BANNED)
print("Features:", FEATURES)
print("No banned column is a feature: OK\n")

print("corr(score, feature):")
for col in ['ctr', 'avg_position', 'impressions', 'impressions_log']:
    r, _ = pearsonr(df[col], df['score'])
    print(f"  {col:16s} {r:+.3f}")
r_imp, _ = pearsonr(df['impressions'], df['score'])
assert abs(r_imp - 0.70) < 0.05

print("\nWindow check: all three features are totals over the same history - no future window, no label window.")

Features: ['ctr', 'avg_position', 'impressions_log']
No banned column is a feature: OK

corr(score, feature):
  ctr              -0.064
  avg_position     -0.123
  impressions      +0.703
  impressions_log  +0.391

Window check: all three features are totals over the same history - no future window, no label window.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

There is no accuracy to compare in an unsupervised lane. Two honest questions instead:

- Do the clusters hold up on unseen clients? Naive split vs grouped split, next to a random-label base rate.

- What do the clusters add on top of the w04 baseline? Same 182,135 items, same score.

In [9]:
# Naive random split vs grouped-by-client split, plus a random-label base rate

def split_sil(mtr, mte):
    sc = StandardScaler().fit(X[mtr]); a, b = sc.transform(X[mtr]), sc.transform(X[mte])
    km = KMeans(n_clusters=K, random_state=SEED, n_init=10, max_iter=300).fit(a)
    return (silhouette_score(a, km.labels_, sample_size=SIL_N, random_state=SEED),
            silhouette_score(b, km.predict(b), sample_size=SIL_N, random_state=SEED))

np.random.seed(SEED)
idx = np.random.permutation(len(df)); cut = int(0.8 * len(df))
naive_train = np.zeros(len(df), bool); naive_train[idx[:cut]] = True
naive_tr, naive_te = split_sil(naive_train, ~naive_train)
grp_tr, grp_te = split_sil(train_mask, test_mask)

both = set(df.loc[naive_train, 'client_hash_id']) & set(df.loc[~naive_train, 'client_hash_id'])
rand_labels = np.random.RandomState(SEED).randint(0, K, size=len(Xs_test))
rand_sil = silhouette_score(Xs_test, rand_labels, sample_size=SIL_N, random_state=SEED)

LOCKED_SPLIT = {'naive': (0.371, 0.375), 'grouped': (0.373, 0.335)}
split_table = pd.DataFrame({
    'split': ['Naive random rows', 'Grouped by client', 'Random labels (base rate)'],
    'train_sil_live': [naive_tr, grp_tr, np.nan],
    'test_sil_live': [naive_te, grp_te, rand_sil],
    'train_sil_locked_w06': [0.371, 0.373, np.nan],
    'test_sil_locked_w06': [0.375, 0.335, np.nan],
})
print(split_table.round(3).to_string(index=False))
print(f"\nClients on BOTH sides of naive split: {len(both)} of {df['client_hash_id'].nunique()} (locked w06: 55)")
assert abs(grp_te - 0.335) < 0.02 and abs(naive_te - 0.375) < 0.02

                    split  train_sil_live  test_sil_live  train_sil_locked_w06  test_sil_locked_w06
        Naive random rows           0.367          0.373                 0.371                0.375
        Grouped by client           0.377          0.336                 0.373                0.335
Random labels (base rate)             NaN         -0.007                   NaN                  NaN

Clients on BOTH sides of naive split: 54 of 59 (locked w06: 55)


In [10]:
# Compare archetypes against the baseline score: profile table, ANOVA effect size, action crosstab

by_arch = df.groupby('archetype').agg(n=('score', 'size'), mean_ctr=('ctr', 'mean'),
                                      mean_position=('avg_position', 'mean'),
                                      mean_impressions=('impressions', 'mean'),
                                      mean_score=('score', 'mean'),
                                      median_score=('score', 'median')).round(4)
print(by_arch.sort_values('mean_score', ascending=False).to_string())

groups = [g['score'].values for _, g in df.groupby('archetype')]
F, p = f_oneway(*groups)
grand = df['score'].mean()
eta2 = sum(len(g) * (g.mean() - grand) ** 2 for g in groups) / ((df['score'] - grand) ** 2).sum()
print(f"\nANOVA on score by archetype: F = {F:,.0f}, p < 1e-300, eta-squared = {eta2:.3f}")
print("Read: clusters and score share inputs - a consistency check, not proof of new signal.")

print("\nBaseline tiers alone:")
print(df['action_label'].value_counts().to_string())
print("\nPlaybook actions (archetype-based):")
print(df['playbook_action'].value_counts().to_string())
print("\nWhere the baseline's HIGH_PRIORITY items land:")
print(pd.crosstab(df['archetype'], df['action_label']).to_string())

                             n  mean_ctr  mean_position  mean_impressions  mean_score  median_score
archetype                                                                                          
High-Reach Opportunity   67701    0.0032        11.7636        24084.8460     32.9837        8.5338
Low-Visibility / Buried  29210    0.0009        45.7875         1458.5729      1.6074        0.5304
Steady / Low Signal      77898    0.0024        13.7349          808.0066      1.2816        0.6268
Already Converting        7326    0.0209        10.2509         3355.3490      0.0000        0.0000

ANOVA on score by archetype: F = 4,258, p < 1e-300, eta-squared = 0.066
Read: clusters and score share inputs - a consistency check, not proof of new signal.

Baseline tiers alone:
action_label
NO_ACTION_CTR_ON_TRACK          160982
FIX_TITLE_META_MONITOR           15882
FIX_TITLE_META_HIGH_PRIORITY      5271

Playbook actions (archetype-based):
playbook_action
LOW_PRIORITY_REVIEW           77898

The naive split looks better on held-out rows only because 54/59 clients sit on both sides. On truly unseen clients: silhouette 0.335 vs 0.373 on train — a small, real gap. Clusters explain only ~6.6% of the score's variance (eta²) and score correlates +0.70 with impressions — the archetypes group what the baseline already ranks; they don't out-rank it. The value-add is different actions, not a better ranking.

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.